In [1]:
%pip install typesafe-sdk

  Using cached typesafe_sdk-0.7.5-py3-none-any.whl.metadata (3.4 kB)
  Using cached anyio-4.16.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached httpx2-2.16.2-py3-none-any.whl.metadata (10.0 kB)
  Using cached pydantic-2.14.4-py3-none-any.whl.metadata (143 kB)
  Using cached tenacity-9.2.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached truststore-0.10.5-py3-none-any.whl.metadata (4.5 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached httpcore2-2.16.2-py3-none-any.whl.metadata (28 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.5-py3-none-any.whl.metadata (2.6 kB)
Using cached typesafe_sdk-0.7.5-py3-none-any.whl (49 kB)
Using cached anyio-4.16.0-py3-none-any.whl (134 kB)
Using cached httpx2-2.16.2-py3-none-any.whl (97 kB)
Using cached httpcore2-2.16.2-py3-none-any.whl (85 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
Using c


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys

print(sys.executable)

C:\Users\zucho\xaifungi-analysis\.venv\Scripts\python.exe


In [3]:
import typesafe_sdk

print("Jev SDK is ready!")

Jev SDK is ready!


In [4]:
%pip install python-dotenv


  Using cached python_dotenv-1.2.4-py3-none-any.whl.metadata (29 kB)
Using cached python_dotenv-1.2.4-py3-none-any.whl (23 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Wczytanie klucza z folderu D_jev
load_dotenv(Path(".env"))

API_KEY = os.getenv("OPENCODE_API_KEY")

# if API_KEY is None:
#     raise ValueError("Nie znaleziono klucza API!")

In [24]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import os
import requests
import json

API_KEY = os.getenv("OPENCODE_API_KEY")

# if not API_KEY:
#     raise ValueError("Brak klucza OPENCODE_API_KEY!")

url = "https://opencode.ai/inference/systemone/v1/systemone"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

payload = {
    "model": "jev-1.13-free",
    "state": (
        "The participant carefully reads the task, "
        "considers different options, and explains "
        "their decision clearly."
    ),
    "questions": {
        "engagement": {
            "type": "score",
            "instructions": (
                "How engaged is the participant in the task?"
            ),
            "criteria": [
                "Low engagement",
                "Moderate engagement",
                "High engagement"
            ]
        }
    }
}

response = requests.post(
    url,
    headers=headers,
    json=payload,
    timeout=60
)

print("HTTP status:", response.status_code)

response.raise_for_status()

result = response.json()

print(json.dumps(result, indent=4, ensure_ascii=False))


HTTP status: 200
{
    "model": "jev-1.13-free",
    "answers": {
        "engagement": {
            "type": "score",
            "score": 2,
            "confidence": 1,
            "legend": {
                "0": "Low engagement",
                "1": "Moderate engagement",
                "2": "High engagement"
            },
            "probabilities": {
                "0": 0,
                "1": 0,
                "2": 1
            }
        }
    },
    "usage": {
        "input_tokens": 320,
        "output_tokens": 19,
        "cost": 0,
        "opencode": {
            "billing_source": "free",
            "list_cost": 0,
            "tokens": {
                "input": 320,
                "output": 19,
                "reasoning": 0,
                "cache_read": 0,
                "cache_write_5m": 0,
                "cache_write_1h": 0
            }
        }
    }
}


In [17]:
import json
from pathlib import Path

transcripts_path = Path("parsed_transcripts.jsonl")

transcripts = []

with open(transcripts_path, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        if line.strip():
            try:
                transcripts.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Błąd w linii {line_number}: {e}")
                raise

print("Liczba wczytanych rekordów:", len(transcripts))
print("Typ danych:", type(transcripts))

if transcripts:
    print("\nKlucze pierwszego rekordu:")
    print(list(transcripts[0].keys()))

    print("\nPierwszy rekord (struktura):")
    for key, value in transcripts[0].items():
        if isinstance(value, str):
            print(f"{key}: tekst ({len(value)} znaków)")
        elif isinstance(value, (list, dict)):
            print(f"{key}: {type(value).__name__}")
        else:
            print(f"{key}: {type(value).__name__}")


Liczba wczytanych rekordów: 1781
Typ danych: <class 'list'>

Klucze pierwszego rekordu:
['source_file', 'slide_id', 'participant_id', 'aggregated_text', 'utterance_count', 'token_count', 'sentence_count', 'first_row', 'last_row']

Pierwszy rekord (struktura):
source_file: tekst (12 znaków)
slide_id: tekst (7 znaków)
participant_id: tekst (2 znaków)
aggregated_text: tekst (623 znaków)
utterance_count: int
token_count: int
sentence_count: int
first_row: int
last_row: int


In [26]:
import os
import json
import requests
from pathlib import Path

# Pobranie klucza API
API_KEY = os.getenv("OPENCODE_API_KEY")

# if not API_KEY:
#     raise ValueError("Brak klucza OPENCODE_API_KEY")

# Wybór pierwszego rekordu
# TODO poniższe do dostosowania na cały zbiór uczestników
record = transcripts[4]

# Tekst uczestnika
participant_text = record["aggregated_text"]

# Dane identyfikacyjne
participant_id = record["participant_id"]
slide_id = record["slide_id"]

# Endpoint i nagłówki
url = "https://opencode.ai/inference/systemone/v1/systemone"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# Definicje pytań
# TODO to jest do zmiany/lepszego dostosowania
questions = {
    "engagement_level": {
        "type": "score",
        "instructions": (
            "Assess the participant's level of engagement "
            "based only on the provided transcript. "
            "Consider active involvement, attention and "
            "willingness to contribute."
        ),
        "criteria": [
            "Low engagement",
            "Moderate engagement",
            "High engagement"
        ]
    },
    "confidence_level": {
        "type": "score",
        "instructions": (
            "Assess the participant's expressed confidence "
            "in their statements and decisions. "
            "Do not assess the model's confidence. "
            "Use only evidence from the transcript."
        ),
        "criteria": [
            "Low expressed confidence",
            "Moderate expressed confidence",
            "High expressed confidence"
        ]
    },
    "communication_style": {
        "type": "score",
        "instructions": (
            "Assess the participant's communication style "
            "based on the structure and elaboration of their "
            "statements, not on the quality of their ideas."
        ),
        "criteria": [
            "Concise and direct communication",
            "Balanced communication",
            "Elaborative and reflective communication"
        ]
    }
}

payload = {
    "model": "jev-1.13-free",
    "state": participant_text,
    "questions": questions
}

# Wywołanie API
response = requests.post(
    url,
    headers=headers,
    json=payload,
    timeout=60
)

print("HTTP status:", response.status_code)

response.raise_for_status()

result = response.json()

# Wyświetlenie wyników
print("\nParticipant:", participant_id)
print("Slide:", slide_id)

print(json.dumps(
    result,
    indent=4,
    ensure_ascii=False
))


HTTP status: 200

Participant: MK_IT_03
Slide: __S2__
{
    "model": "jev-1.13-free",
    "answers": {
        "engagement_level": {
            "type": "score",
            "score": 1.22,
            "confidence": 0.56,
            "legend": {
                "0": "Low engagement",
                "1": "Moderate engagement",
                "2": "High engagement"
            },
            "probabilities": {
                "0": 0.03,
                "1": 0.71,
                "2": 0.26
            }
        },
        "confidence_level": {
            "type": "score",
            "score": 0.21,
            "confidence": 0.67,
            "legend": {
                "0": "Low expressed confidence",
                "1": "Moderate expressed confidence",
                "2": "High expressed confidence"
            },
            "probabilities": {
                "0": 0.79,
                "1": 0.2,
                "2": 0.01
            }
        },
        "communication_style": {
     

In [27]:
# test zapisu odpowiedzi

pilot_result = {
    "participant_id": participant_id,
    "slide_id": slide_id,
    "source_file": record["source_file"],
    "scores": result["answers"],
    "usage": result.get("usage", {})
}

with open(
    "jev_pilot.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        pilot_result,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Zapisano wynik do jev_pilot.json")

Zapisano wynik do jev_pilot.json


## Run JEV on all participant's scripts

In [28]:

def analyze_with_jev(record, api_key):
    """
    Analyze one participant-slide transcript using Jev.
    Returns a structured result.
    """

    url = "https://opencode.ai/inference/systemone/v1/systemone"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    questions = {
        "engagement_level": {
            "type": "score",
            "instructions": (
                "Assess the participant's level of engagement "
                "based only on the transcript. Consider active "
                "involvement, attention and willingness to contribute."
            ),
            "criteria": [
                "Low engagement",
                "Moderate engagement",
                "High engagement"
            ]
        },
        "confidence_level": {
            "type": "score",
            "instructions": (
                "Assess the participant's expressed confidence "
                "in their statements and decisions. Do not assess "
                "the model's confidence."
            ),
            "criteria": [
                "Low expressed confidence",
                "Moderate expressed confidence",
                "High expressed confidence"
            ]
        },
        "communication_style": {
            "type": "score",
            "instructions": (
                "Assess the participant's communication style "
                "based on structure and elaboration of statements."
            ),
            "criteria": [
                "Concise and direct communication",
                "Balanced communication",
                "Elaborative and reflective communication"
            ]
        }
    }

    payload = {
        "model": "jev-1.13-free",
        "state": record["aggregated_text"],
        "questions": questions
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=60
    )

    response.raise_for_status()

    result = response.json()

    scores = {}

    for dimension, answer in result["answers"].items():
        score = answer["score"]

        scores[dimension] = {
            "score": score,
            "normalized_score": score / 2,
            "confidence": answer["confidence"],
            "probabilities": answer["probabilities"]
        }

    return {
        "participant_id": record["participant_id"],
        "slide_id": record["slide_id"],
        "source_file": record["source_file"],
        "utterance_count": record["utterance_count"],
        "token_count": record["token_count"],
        "scores": scores,
        "usage": result.get("usage", {})
    }

In [31]:

import json
import time
from pathlib import Path

# Pliki wyjściowe
results_path = Path("jev_results.jsonl")
errors_path = Path("jev_errors.jsonl")

# Identyfikator rekordu - participant + slide + source
def record_key(record):
    return (
        str(record.get("source_file", "")),
        str(record.get("participant_id", "")),
        str(record.get("slide_id", ""))
    )

# Wczytaj już przetworzone rekordy (checkpoint)
processed_keys = set()

if results_path.exists():
    with open(results_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                saved = json.loads(line)
                processed_keys.add(record_key(saved))

print("Już przetworzone rekordy:", len(processed_keys))

# Wybierz rekordy, których jeszcze nie analizowaliśmy
pending = [
    r for r in transcripts
    if record_key(r) not in processed_keys
]

print("Pozostało do analizy:", len(pending))


Już przetworzone rekordy: 0
Pozostało do analizy: 1781


In [33]:
BATCH_SIZE = len(pending)
DELAY_SECONDS = 1

batch = pending[:BATCH_SIZE]

success_count = 0
error_count = 0

for i, record in enumerate(batch, start=1):

    key = record_key(record)

    print(
        f"[{i}/{len(batch)}] "
        f"Participant={record['participant_id']} | "
        f"Slide={record['slide_id']}"
    )

    try:
        output = analyze_with_jev(record, API_KEY)

        # Zapis od razu po sukcesie (checkpoint)
        with open(results_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(output, ensure_ascii=False) + "\n")

        processed_keys.add(key)
        success_count += 1

        print("  -> OK")

    except Exception as e:

        error_entry = {
            "participant_id": record.get("participant_id"),
            "slide_id": record.get("slide_id"),
            "source_file": record.get("source_file"),
            "error": str(e)
        }

        with open(errors_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(error_entry, ensure_ascii=False) + "\n")

        error_count += 1
        print("  -> ERROR:", str(e))

    time.sleep(DELAY_SECONDS)

print("\nBatch zakończony.")
print("Sukcesy:", success_count)
print("Błędy:", error_count)
print("Łącznie zapisanych wyników:", len(processed_keys))


[1/1781] Participant=MK | Slide=__S00__
  -> OK
[2/1781] Participant=MK_IT_03 | Slide=__S00__
  -> OK
[3/1781] Participant=MK_IT_03 | Slide=__S1__
  -> OK
[4/1781] Participant=MK | Slide=__S1__
  -> OK
[5/1781] Participant=MK_IT_03 | Slide=__S2__
  -> OK
[6/1781] Participant=MK_IT_03 | Slide=__S3__
  -> OK
[7/1781] Participant=MK_IT_03 | Slide=__S4__
  -> OK
[8/1781] Participant=MK | Slide=__S4__
  -> OK
[9/1781] Participant=MK_IT_03 | Slide=__S5__
  -> OK
[10/1781] Participant=MK_IT_03 | Slide=__S6__
  -> OK
[11/1781] Participant=MK_IT_03 | Slide=__S7__
  -> OK
[12/1781] Participant=MK | Slide=__S7__
  -> OK
[13/1781] Participant=MK_IT_03 | Slide=__S8__
  -> OK
[14/1781] Participant=MK_IT_03 | Slide=__S9__
  -> OK
[15/1781] Participant=MK_IT_03 | Slide=__S10__
  -> OK
[16/1781] Participant=MK_IT_03 | Slide=__S11__
  -> OK
[17/1781] Participant=MK_IT_03 | Slide=__S13__
  -> OK
[18/1781] Participant=MK_IT_03 | Slide=__S14__
  -> OK
[19/1781] Participant=MK | Slide=__S15__
  -> OK
[20/17

KeyboardInterrupt: 

### Quality check

In [34]:

import json
from pathlib import Path

results_path = Path("jev_results.jsonl")

with open(results_path, "r", encoding="utf-8") as f:
    saved_results = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

expected_dimensions = {
    "engagement_level",
    "confidence_level",
    "communication_style"
}

issues = []
seen_keys = set()

for i, item in enumerate(saved_results):

    key = (
        item.get("source_file"),
        item.get("participant_id"),
        item.get("slide_id")
    )

    if key in seen_keys:
        issues.append(f"Duplicate record at index {i}: {key}")
    seen_keys.add(key)

    scores = item.get("scores", {})

    if set(scores.keys()) != expected_dimensions:
        issues.append(f"Missing dimensions at index {i}")

    for dimension, data in scores.items():

        score = data.get("score")
        normalized = data.get("normalized_score")
        confidence = data.get("confidence")
        probabilities = data.get("probabilities", {})

        if score is None or not 0 <= score <= 2:
            issues.append(f"Invalid score: {i}, {dimension}")

        if normalized is None or not 0 <= normalized <= 1:
            issues.append(f"Invalid normalized score: {i}, {dimension}")

        if confidence is None or not 0 <= confidence <= 1:
            issues.append(f"Invalid confidence: {i}, {dimension}")

        if probabilities:
            prob_sum = sum(probabilities.values())

            if abs(prob_sum - 1.0) > 0.02:
                issues.append(
                    f"Probabilities do not sum to 1: {i}, {dimension}"
                )

print("Liczba rekordów:", len(saved_results))
print("Liczba unikalnych kluczy:", len(seen_keys))
print("Liczba problemów:", len(issues))

for issue in issues[:20]:
    print("-", issue)

if not issues:
    print("QC PASSED - wszystkie kontrole zakończone pomyślnie!")


Liczba rekordów: 59
Liczba unikalnych kluczy: 39
Liczba problemów: 20
- Duplicate record at index 20: ('MK_IT_03.csv', 'MK', '__S00__')
- Duplicate record at index 21: ('MK_IT_03.csv', 'MK_IT_03', '__S00__')
- Duplicate record at index 22: ('MK_IT_03.csv', 'MK_IT_03', '__S1__')
- Duplicate record at index 23: ('MK_IT_03.csv', 'MK', '__S1__')
- Duplicate record at index 24: ('MK_IT_03.csv', 'MK_IT_03', '__S2__')
- Duplicate record at index 25: ('MK_IT_03.csv', 'MK_IT_03', '__S3__')
- Duplicate record at index 26: ('MK_IT_03.csv', 'MK_IT_03', '__S4__')
- Duplicate record at index 27: ('MK_IT_03.csv', 'MK', '__S4__')
- Duplicate record at index 28: ('MK_IT_03.csv', 'MK_IT_03', '__S5__')
- Duplicate record at index 29: ('MK_IT_03.csv', 'MK_IT_03', '__S6__')
- Duplicate record at index 30: ('MK_IT_03.csv', 'MK_IT_03', '__S7__')
- Duplicate record at index 31: ('MK_IT_03.csv', 'MK', '__S7__')
- Duplicate record at index 32: ('MK_IT_03.csv', 'MK_IT_03', '__S8__')
- Duplicate record at index 3

In [38]:
BATCH_SIZE = 20
DELAY_SECONDS = 1

# Świeże dane z pliku wynikowego
saved_results = [
    json.loads(line)
    for line in results_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

processed_keys = {
    record_key(item)
    for item in saved_results
}

# Dynamiczne wyznaczenie brakujących rekordów
pending = [
    r for r in transcripts
    if record_key(r) not in processed_keys
]

batch = pending[:BATCH_SIZE]

success_count = 0
error_count = 0

print(f"Rozpoczynam batch: {len(batch)} rekordów")
print(f"Pozostało łącznie: {len(pending)}")

for i, record in enumerate(batch, start=1):

    key = record_key(record)

    print(
        f"[{i}/{len(batch)}] "
        f"Participant={record['participant_id']} | "
        f"Slide={record['slide_id']}"
    )

    try:
        output = analyze_with_jev(record, API_KEY)

        with open(results_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(output, ensure_ascii=False) + "\n")

        processed_keys.add(key)
        success_count += 1
        print("  -> OK")

    except Exception as e:
        error_count += 1

        error_entry = {
            "source_file": record["source_file"],
            "participant_id": record["participant_id"],
            "slide_id": record["slide_id"],
            "error": str(e)
        }

        with open(errors_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(error_entry, ensure_ascii=False) + "\n")

        print(f"  -> ERROR: {e}")

    if i < len(batch):
        time.sleep(DELAY_SECONDS)

print("\nBatch zakończony.")
print("Sukcesy:", success_count)
print("Błędy:", error_count)
print("Zapisane łącznie:", len(processed_keys))
print("Pozostało:", len(set(input_keys) - processed_keys))


Rozpoczynam batch: 20 rekordów
Pozostało łącznie: 1742
[1/20] Participant=MK_IT_06 | Slide=__S11__
  -> OK
[2/20] Participant=MK | Slide=__S11__
  -> OK
[3/20] Participant=MK_IT_06 | Slide=__S12__
  -> OK
[4/20] Participant=MK_IT_06 | Slide=__S13__
  -> OK
[5/20] Participant=MK | Slide=__S15__
  -> OK
[6/20] Participant=MK | Slide=__S88__
  -> OK
[7/20] Participant=MK_IT_06 | Slide=__S88__
  -> OK
[8/20] Participant=MK | Slide=__S99__
  -> OK
[9/20] Participant=MK_IT_06 | Slide=__S99__
  -> OK
[10/20] Participant=MK | Slide=__S00__
  -> OK
[11/20] Participant=MK_SSH_02 | Slide=__S00__
  -> OK
[12/20] Participant=MK_SSH_02 | Slide=__S1__
  -> OK
[13/20] Participant=MK_SSH_02 | Slide=__S2__
  -> OK
[14/20] Participant=MK_SSH_02 | Slide=__S3__
  -> OK
[15/20] Participant=MK_SSH_02 | Slide=__S4__
  -> OK
[16/20] Participant=MK | Slide=__S4__
  -> OK
[17/20] Participant=MK_SSH_02 | Slide=__S5__
  -> OK
[18/20] Participant=MK_SSH_02 | Slide=__S6__
  -> OK
[19/20] Participant=MK | Slide=__S6_